<img src="http://www.cidaen.es/assets/img/cidaenNB.png" alt="Logo CiDAEN" align="right">


<br><br><br>
<h2><font color="#00586D" size=4>Módulo 12: Arquitecturas y procesos Big Data</font></h2>



<h1><font color="#00586D" size=5>Capstone 12. Parte 2: Uso del modelo de sentiment

<br><br><br>
<div style="text-align: right">
<font color="#00586D" size=3>Enrique González, Jacinto Arias</font><br>
<font color="#00586D" size=3>Máster en Ciencia de Datos e Ingeniería de Datos en la Nube</font><br>
<font color="#00586D" size=3>Universidad de Castilla-La Mancha</font>




</div>

<a id="indice"></a>
<h2><font color="#00586D" size=5>Índice</font></h2>


* [1. Uso batch del modelo serializado](#section1)

A continuación vamos a preparar el entorno. Recordad que tenéis que ingresar vuestras credenciales de AWS Academy en el fichero .env antes de levantar Jupyter Lab.

In [5]:
%iam_role arn:aws:iam::270830193283:role/LabRole
%region us-east-1
%number_of_workers 2
%idle_timeout 30

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
It looks like there is a newer version of the kernel available. The latest version is 1.0.9 and you have 1.0.4 installed.
Please run `pip install --upgrade aws-glue-sessions` to upgrade your kernel
Current iam_role is None
iam_role has been set to arn:aws:iam::270830193283:role/LabRole.
Previous region: None
Setting new region to: us-east-1
Region is set to: us-east-1
Previous number of workers: None
Setting new number of workers to: 2
Current idle_timeout is None minutes.
idle_timeout has been set to 30 minutes.


In [1]:
s3_bucket = "s3://cidaen-capstone-12-1"

Trying to create a Glue session for the kernel.
Session Type: etl
Worker Type: G.1X
Number of Workers: 2
Session ID: c8680077-3f8c-489d-9933-cdb847fba005
Applying the following default arguments:
--glue_kernel_version 1.0.4
--enable-glue-datacatalog true
Waiting for session c8680077-3f8c-489d-9933-cdb847fba005 to get into ready status...
Session c8680077-3f8c-489d-9933-cdb847fba005 has been created.



---

<a id="section3"></a>
## <font color="#00586D"> 1. Uso batch del modelo serializado</font>
<br>

Vamos a cargar el modelo entrenado en la primera parte del capstone y lo vamos a aplicar en batch sobre el dataset que habíamos guardado previamente.

---
### <font color="#004D7F"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#004D7F"></i> Tarea 10: Uso del modelo en batch </font>
<br>


Carga el modelo que hemos guardado en la parte 1 y aplicalo a los datos de test que también guardamos en esa parte. Evalúalo usando el área bajo la curva ROC.  

In [2]:
from pyspark.ml import PipelineModel
from pyspark.ml.evaluation import BinaryClassificationEvaluator


In [3]:
model_loaded = PipelineModel.load("s3://cidaen-capstone-12-1/word2vec_model/")
test_df_loaded = spark.read.parquet("s3://cidaen-capstone-12-1/electronics_test/")
predictions = model_loaded.transform(test_df_loaded)

evaluator = BinaryClassificationEvaluator(labelCol="sentiment", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
auc_loaded = evaluator.evaluate(predictions)

print(f"AUC del modelo cargado: {auc_loaded:.4f}")



AUC del modelo cargado: 0.9116


In [8]:
import pandas as pd
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt


roc_df = predictions.select("probability", "sentiment").toPandas()


roc_df["prob_positive"] = roc_df["probability"].apply(lambda x: float(x[1]))


fpr, tpr, thresholds = roc_curve(roc_df["sentiment"], roc_df["prob_positive"])
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color="blue", lw=2, label=f"ROC curve (AUC = {roc_auc:.2f})")
plt.plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Curva ROC - Modelo de Sentiment (Test)")
plt.legend(loc="lower right")
plt.savefig("roc_curve.png") 


In [16]:
import boto
s3 = boto3.client('s3')
s3.upload_file('roc_curve.png', 'cidaen-capstone-12-1', 'images/roc_curve.png')




In [21]:
from PIL import Image as PILImage
import matplotlib.pyplot as plt

img = PILImage.open("roc_curve.png")
plt.imshow(img)
plt.axis('off')
plt.show()

In [22]:
import os
print(os.listdir())

['hsperfdata_spark', 'glue-default.conf', 'glue-override.conf', 'log4j2.properties', '.log4j2.properties.crc', 'exportCustomerEnvVariablesCmd', 'exportInternalEnvVariablesCmd', 'launchcmd', '5029492955850623067', 'blockmgr-e033ae39-ccd4-4817-8165-2c4969fab639', 'spark-d2baad95-16df-46c9-b667-8d72bed4010e', 'liblz4-java-911791017396719452.so.lck', 'liblz4-java-911791017396719452.so', 'hadoop-spark', 'roc_curve.png', 'glue-app-insights-logs', 'glue-exception-analysis-logs', 'glue_app_analyzer_rules', 'spark-event-logs', 'aws_glue_custom_connector_python', '.ICE-unix', '.Test-unix', '.X11-unix', '.XIM-unix', '.font-unix']


No entiendo porque no se muestra la imagen si esta en el directorio...